In [1]:
import os
import tarfile
import tensorflow as tf
from metaflow import Flow, S3

In [2]:
FLOW_NAME = "MultiNodeTensorFlow"
model_unzip_path = "model"

flow = Flow(FLOW_NAME)
run = flow.latest_successful_run
_s3 = S3(run=run)
s3obj = _s3.get(run.data.local_tar_name)

_file = tarfile.open(s3obj.path)
_file.extractall(model_unzip_path)
_file.close()
_s3.close()

In [3]:
load_path = os.path.join(os.getcwd(), model_unzip_path, run.data.local_model_dir)
model = tf.keras.models.load_model(load_path)

In [4]:
(x_train, y_train), (x_valid, y_valid) = tf.keras.datasets.mnist.load_data()
probs = model.predict(x_valid)
preds = probs.argmax(axis=1)
correct_pred_ct = (preds == y_valid).sum()
accuracy = correct_pred_ct / preds.shape[0]
print("Accuracy: %.2f%%" % (accuracy * 100.0))

313/313 [==============================] - 1s 3ms/step
Accuracy: 88.20%
